In [ ]:
%pip install "numpy==2.3.5" "pandas==2.2.3" "matplotlib==3.10.8"

# IEEE THMS — Task Mining: Publication-Ready Analytics

**45 figures and 4 publication-ready tables**, with PNG-only image export (8 cm wide at 300 dpi); accompanying numerical results are saved separately as CSV. Analyses include **only completed sessions from participants with IDs matching `P###`**. Paired comparisons and confidence intervals respect the within-subject design.


## 1. Configuration and loading the Task Mining archive

Place `analytics.ipynb` and the source ZIP file in the same folder. If more than one ZIP file is present, explicitly set `ZIP_NAME` below. The notebook creates `output/figures/` and `output/tables/` automatically.


In [ ]:
from pathlib import Path
import zipfile, json, hashlib, datetime, math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Resolve input files relative to the notebook's working directory (standard Jupyter launch).
BASE_DIR = Path.cwd().resolve()
ZIP_NAME = None             # For example, 'task-mining(3).zip'; None requires exactly one nearby ZIP
OUTPUT_DIR = BASE_DIR / "output"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
SEED = 20261020
BOOTSTRAP_N = 4000
PERMUTATIONS = 40000
FIG_WIDTH_CM = 8.0
DPI = 300

for folder in (FIG_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Remove only obsolete filenames from older notebook releases.
# The current Figs. 34–36 use signed win margins rather than unsigned win rates.
for level in (1, 2, 3):
    obsolete = FIG_DIR / f'fig{33 + level:02d}_paired_win_probability_c{level}.png'
    if obsolete.exists():
        obsolete.unlink()

archives = sorted(p for p in BASE_DIR.glob('*.zip') if p.is_file())
if ZIP_NAME:
    archive_path = BASE_DIR / ZIP_NAME
elif len(archives) == 1:
    archive_path = archives[0]
else:
    raise FileNotFoundError(f'Expected exactly one ZIP in {BASE_DIR}; found {len(archives)}. Set ZIP_NAME explicitly.')
if not archive_path.is_file():
    raise FileNotFoundError(f'ZIP archive not found: {archive_path}')

with zipfile.ZipFile(archive_path) as z:
    name_map = {Path(name).name: name for name in z.namelist() if not name.endswith('/')}
    required = ['registry.csv','trials.csv','events.csv','attempts.csv',
                'interpretations.csv','m4_transcriptions.csv','agent_runs.csv','protocol.json']
    missing = [name for name in required if name not in name_map]
    if missing:
        raise ValueError(f'Required export components are missing: {missing}')
    def csv_from_zip(filename):
        return pd.read_csv(z.open(name_map[filename]), low_memory=False)
    registry = csv_from_zip('registry.csv')
    trials_all = csv_from_zip('trials.csv')
    attempts = csv_from_zip('attempts.csv')
    events = csv_from_zip('events.csv')
    interpretations = csv_from_zip('interpretations.csv')
    transcriptions = csv_from_zip('m4_transcriptions.csv')
    agent_runs = csv_from_zip('agent_runs.csv')
    protocol = json.load(z.open(name_map['protocol.json']))

archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Source archive:', archive_path.name)
print('Rows: registry', len(registry), 'trials', len(trials_all), 'events', len(events),
      'attempts', len(attempts))
print('ASR model in manifest:', protocol.get('manifest',{}).get('m4_asr',{}).get('model','not declared'))

## 2. Participant inclusion, session completeness, and measurement validation

**Inclusion requires both:** (1) a participant code that exactly matches `P[0-9]{3}` and (2) `complete=True` in `registry.csv`. Each eligible session must contain exactly 15 terminal experimental trials (M1–M5 × C1–C3). Test accounts, incomplete sessions, and practice trials are excluded, but unsuccessful **experimental trials in completed sessions remain included**.


In [ ]:
import re

MODE_ORDER = ['M1','M2','M3','M4','M5']
PARTICIPANT_CODE_PATTERN = r'P[0-9]{3}'  # Full match: eligible participant IDs such as P001, P002, ...
LEVEL_ORDER = [1,2,3]

def require_columns(frame, expected, name):
    absent = set(expected) - set(frame.columns)
    if absent:
        raise ValueError(f'{name}: missing required columns: {sorted(absent)}')

def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    parsed = series.astype(str).str.strip().str.lower().map(
        {'true':True,'1':True,'false':False,'0':False})
    if parsed.isna().any():
        raise ValueError('Invalid True/False values in the export')
    return parsed.astype(bool)

require_columns(registry,['session_id','participant_code','complete'], 'registry')
require_columns(trials_all, ['session_id','trial_id','participant_code','kind','mode','level',
                             'status','Tcorrect_analysis_ms','Tcorrect_actual_ms',
                             'Tuser_active_ms','A1_analysis','A1_actual',
                             'Nretry_analysis','Nretry_actual','attempts','elapsed_ms',
                             'gave_up','end_reason'], 'trials')
if trials_all['trial_id'].duplicated().any():
    raise ValueError('Duplicate trial_id in trials.csv')
registry['complete'] = to_bool(registry['complete'])
trials_all['level'] = pd.to_numeric(trials_all['level'],errors='raise').astype(int)
for col in ['Tcorrect_analysis_ms','Tcorrect_actual_ms','Tuser_active_ms',
            'A1_analysis','A1_actual','Nretry_actual','Nretry_analysis','elapsed_ms','attempts']:
    trials_all[col] = pd.to_numeric(trials_all[col],errors='coerce')
trials_all['gave_up'] = to_bool(trials_all['gave_up'])
if not set(trials_all.status.unique()) <= {'correct','incomplete','active','pending'}:
    raise ValueError('Unknown trial status value')

# Strict double filter: a P### participant code AND complete=True.
# Incomplete P### sessions do not enter the analytical population.
# Test accounts (e.g., z###/M###) are excluded even if complete=True.
registry['participant_code'] = registry['participant_code'].astype('string').str.strip()
trials_all['participant_code'] = trials_all['participant_code'].astype('string').str.strip()
is_real_code = registry['participant_code'].str.fullmatch(PARTICIPANT_CODE_PATTERN, na=False)
completed_registry = registry.loc[is_real_code & registry['complete'],
                                  ['session_id','participant_code']].copy()
if completed_registry.empty:
    raise ValueError('No completed P### participant sessions found in registry.csv')

if completed_registry['session_id'].duplicated().any():
    raise ValueError('Duplicate session_id in registry')
trials = trials_all.loc[
    (trials_all['kind']=='experiment') &
    trials_all['session_id'].isin(completed_registry['session_id'])
].copy()
# Integrity check: each trial must match the participant code registered for its session_id.
registry_codes = completed_registry.set_index('session_id')['participant_code']
trial_expected_codes = trials['session_id'].map(registry_codes)
if not trials['participant_code'].eq(trial_expected_codes).all():
    raise ValueError('Participant codes differ between trials.csv and registry.csv')
if not trials['participant_code'].str.fullmatch(PARTICIPANT_CODE_PATTERN, na=False).all():
    raise ValueError('A non-P### participant code survived inclusion filtering')


expected_pairs = {(m,l) for m in MODE_ORDER for l in LEVEL_ORDER}
for sid, group in trials.groupby('session_id'):
    present = list(zip(group['mode'],group['level']))
    if len(present)!=15 or set(present)!=expected_pairs or len(set(present))!=15:
        raise ValueError(f'Session {sid}: expected a complete balanced set of 15 trials')
    if not group.status.isin(['correct','incomplete']).all():
        raise ValueError(f'Session {sid}: nonterminal trials found despite complete=True')
if trials['participant_code'].nunique()!=trials['session_id'].nunique():
    raise ValueError('Multiple completed sessions per participant; an explicit selection rule is needed')
if len(trials)!=len(completed_registry)*15:
    raise ValueError('Completed-session and trial counts do not agree')
if trials[['Tcorrect_analysis_ms','Tuser_active_ms','A1_analysis','Nretry_analysis']].isna().any().any():
    raise ValueError('A completed trial lacks a required analysis metric')
if not (trials.loc[trials.status=='incomplete','Tcorrect_actual_ms'].isna()).all():
    raise ValueError('Unsuccessful trials must not have Tcorrect_actual_ms')
if trials.loc[trials.status=='correct','Tcorrect_actual_ms'].isna().any():
    raise ValueError('A successful trial is missing Tcorrect_actual_ms')
if not set(trials['mode']) <= set(MODE_ORDER):
    raise ValueError('Unexpected interaction mode')

subjects = sorted(trials['participant_code'].unique().tolist())
N_SUBJECTS = len(subjects)
trials['mode'] = pd.Categorical(trials['mode'],categories=MODE_ORDER,ordered=True)
trials['success'] = (trials['status']=='correct').astype(int)
trials['time_min'] = trials['Tcorrect_analysis_ms']/60000.0
trials['actual_time_min'] = trials['Tcorrect_actual_ms']/60000.0
trials['active_min'] = trials['Tuser_active_ms']/60000.0
trials['A1'] = trials['A1_analysis']
trials['retries'] = trials['Nretry_analysis']
trials['gaveup_flag'] = (trials['status']=='incomplete') & trials['gave_up']
trials['timeout_flag'] = (trials['status']=='incomplete') & (trials['end_reason'].isin(['time_limit','timeout']))

def sample_flow_table():
    return pd.DataFrame([
        ('All sessions in ZIP',len(registry)),
        ('Excluded non-P### sessions',int((~is_real_code).sum())),
        ('P### sessions not complete',int((is_real_code & ~registry.complete).sum())),
        ('Included completed P### sessions',len(completed_registry)),
        ('Unique included participants',N_SUBJECTS),
        ('Included experimental trials',len(trials)),
        ('Successful trials',int(trials.success.sum())),
        ('Unsuccessful trials',int((1-trials.success).sum())),
    ],columns=['Item','N'])

flow_table = sample_flow_table()
print(flow_table.to_string(index=False))
print('Complete trials per mode × level:',trials.groupby(['mode','level'],observed=True).size().unique().tolist())
print('Included codes:', ', '.join(subjects))
assert all(re.fullmatch(PARTICIPANT_CODE_PATTERN, x) for x in subjects)


## 3. Publication styling and export functions (exactly 8 cm)

All plots and rendered tables use a physical width of 8 cm at 300 dpi. Plot labels, legends, annotations, and messages are in English. A high-contrast color palette replaces grayscale heatmaps and low-opacity symbols; mode colors remain consistent across all figures. CSV exports contain numerical summaries; no PDF figures are generated.


In [ ]:
FIG_WIDTH_IN = FIG_WIDTH_CM / 2.54
COLORS = {'M1':'#2556A8','M2':'#C98100','M3':'#C64247',
          'M4':'#008F78','M5':'#823B9A'}
MARKERS = {'M1':'o','M2':'s','M3':'^','M4':'D','M5':'v'}
LINE_STYLES = {'M1':'-','M2':'--','M3':'-.','M4':':','M5':'-'}

mpl.rcParams.update({
    'font.family':'DejaVu Sans',
    'font.size':7.3,'axes.labelsize':7.7,'axes.titlesize':8,
    'legend.fontsize':6.5,'xtick.labelsize':6.7,'ytick.labelsize':6.7,
    'axes.linewidth':0.55,'lines.linewidth':1.2,'lines.markersize':4,
    'xtick.major.width':0.5,'ytick.major.width':0.5,
    'grid.linewidth':0.45,'savefig.dpi':DPI,'figure.dpi':DPI,
    'figure.facecolor':'white','axes.facecolor':'white',
})

figure_paths=[]
table_paths=[]
def newfig(height_cm=6.5):
    fig,ax = plt.subplots(figsize=(FIG_WIDTH_IN,height_cm/2.54),dpi=DPI)
    return fig,ax

def scientific_axes(ax,grid='y'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_axisbelow(True)
    if grid: ax.grid(axis=grid,alpha=0.16,color='#8793A6')

def save_figure(fig,stem):
    # Do not use bbox_inches='tight': it changes the physical image width from 8 cm.
    # Use nearest integral pixel width: 945 px at 300 dpi = 8.001 cm.
    pixel_width=round(FIG_WIDTH_IN*DPI)
    fig.set_size_inches((pixel_width+0.01)/DPI,fig.get_size_inches()[1],forward=True)
    filename=FIG_DIR/f'{stem}.png'
    fig.savefig(filename,dpi=DPI,facecolor='white',bbox_inches=None)
    figure_paths.append(filename)
    plt.close(fig)
    return filename

def save_csv(df,stem):
    path=TABLE_DIR/f'{stem}.csv'
    df.to_csv(path,index=False,encoding='utf-8-sig',float_format='%.6f')
    table_paths.append(path)
    return path

rng=np.random.default_rng(SEED)
def bootstrap_stat(values,stat='median',n=BOOTSTRAP_N):
    v=np.asarray(values,dtype=float)
    v=v[np.isfinite(v)]
    if not len(v): return (np.nan,np.nan,np.nan)
    if stat=='median': op=np.median
    elif stat=='mean': op=np.mean
    else: raise ValueError(stat)
    draws=rng.integers(0,len(v),size=(n,len(v)))
    bootstrap=op(v[draws],axis=1)
    return float(op(v)),float(np.quantile(bootstrap,0.025)),float(np.quantile(bootstrap,0.975))

def metric_matrix(column):
    pivot=trials.pivot(index='participant_code',columns=['mode','level'],values=column)
    target=pd.MultiIndex.from_product([MODE_ORDER,LEVEL_ORDER],names=['mode','level'])
    pivot=pivot.reindex(index=subjects,columns=target)
    if pivot.isna().any().any():
        raise ValueError(f'Incomplete mode-by-complexity matrix for {column}')
    return pivot

def curve_chart(column,stem,ylabel,scale=1.,height_cm=7.0,ylim=None,stat='median'):
    matrix=metric_matrix(column)
    fig,ax=newfig(height_cm)
    for mode in MODE_ORDER:
        values=[matrix[(mode,l)].to_numpy(dtype=float)/scale for l in LEVEL_ORDER]
        agg=np.asarray([bootstrap_stat(v,stat=stat) for v in values])
        ax.errorbar(LEVEL_ORDER,agg[:,0],
                    yerr=np.vstack((agg[:,0]-agg[:,1],agg[:,2]-agg[:,0])),
                    color=COLORS[mode],marker=MARKERS[mode],ls=LINE_STYLES[mode],
                    linewidth=1.05,markersize=3.8,capthick=0.75,capsize=1.6,
                    elinewidth=0.75,label=mode)
    scientific_axes(ax)
    ax.set_xticks(LEVEL_ORDER,['C1','C2','C3'])
    ax.set_xlabel('Task complexity')
    ax.set_ylabel(ylabel)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.legend(ncol=3,loc='upper left',frameon=False,handlelength=1.75,
              columnspacing=0.9,borderaxespad=0.3)
    fig.subplots_adjust(left=0.19,right=0.97,bottom=0.16,top=0.96)
    return save_figure(fig,stem)

def heatmap(column,stem,percent=True):
    cell=trials.groupby(['mode','level'],observed=True)[column].agg(['mean','count'])
    data=np.asarray([[cell.loc[(m,l),'mean'] for l in LEVEL_ORDER] for m in MODE_ORDER])
    n=np.asarray([[cell.loc[(m,l),'count'] for l in LEVEL_ORDER] for m in MODE_ORDER])
    fig,ax=newfig(6.7)
    im=ax.imshow(data*100 if percent else data,cmap='YlGnBu',vmin=0,vmax=100 if percent else None,
                 origin='upper',aspect='auto')
    ax.set_xticks(range(3),['C1','C2','C3'])
    ax.set_yticks(range(5),MODE_ORDER)
    ax.tick_params(axis='both',length=0)
    ax.set_xlabel('Task complexity')
    for i in range(5):
        for j in range(3):
            p=data[i,j]*100 if percent else data[i,j]
            rgba=im.cmap(im.norm(p))
            brightness=0.2126*rgba[0]+0.7152*rgba[1]+0.0722*rgba[2]
            color='white' if brightness<0.48 else '#111111'
            ax.text(j,i, f'{p:.0f}%' if percent else f'{p:.2f}',
                    color=color,ha='center',va='center',fontsize=7.9,fontweight='bold')
            ax.text(j,i+0.27,f'n={n[i,j]}',color=color,ha='center',va='center',fontsize=5.6)
    fig.subplots_adjust(left=0.21,right=0.95,bottom=0.16,top=0.96)
    return save_figure(fig,stem)

print(f'Image width: {FIG_WIDTH_CM:.2f} cm = {FIG_WIDTH_IN:.4f} in = {int(round(FIG_WIDTH_IN*DPI))} px @ {DPI} dpi')

## 4. Primary figures: completion time, success, and retries

Time-series panels show **participant-level medians** with 95% bootstrap confidence intervals. Retry counts in Fig. 05 are shown as **means** to retain the influence of infrequent failures. Heatmaps report the proportion of trials in each mode–complexity cell (N = number of included participants; N = 37 in the supplied export). `Tcorrect_analysis_ms` includes protocol-defined penalties, whereas `Tuser_active_ms` is an activity estimate derived from logged events.


In [ ]:
curve_chart('Tcorrect_analysis_ms','fig01_correct_time_by_complexity',
            'Analysis time incl. penalty (min)',scale=60000)
heatmap('success','fig02_success_rate')
heatmap('A1','fig03_first_attempt_accuracy')
curve_chart('Tuser_active_ms','fig04_active_user_time',
            'Recorded active time (min)',scale=60000)
curve_chart('Nretry_analysis','fig05_retry_count_analysis',
            'Mean retries incl. penalty',scale=1,stat='mean')
print('Generated figures 01–05')

## 5. Trial outcomes and paired contrasts

For each of M3/M4/M5, the difference from the GUI baseline (M1) is evaluated at C1 and C3 separately. Negative differences indicate an advantage of the comparison mode. The **interaction contrast** is `(Mx − M1) at C3 − (Mx − M1) at C1`; negative values indicate a larger time advantage at higher complexity. Statistical inference uses a two-sided sign-flip permutation test on participant-level mean contrasts, with Holm adjustment across the three specified comparisons. **Design limitation:** C1, C2, and C3 were always presented in that order within each block.


In [ ]:
# 06 — Trial outcomes by mode (N_SUBJECTS × 3 trials per mode).
outcomes=[]
for m in MODE_ORDER:
    ss=trials.loc[trials['mode']==m]
    n=len(ss)
    win=int((ss['status']=='correct').sum())
    give=int(ss['gaveup_flag'].sum())
    timeout=int(ss['timeout_flag'].sum())
    other=n-win-give-timeout
    outcomes.append(dict(mode=m,n=n,correct=win,gave_up=give,time_limit=timeout,other_incomplete=other))
outcome_df=pd.DataFrame(outcomes)
save_csv(outcome_df,'table_outcomes_by_mode')
fig,ax=newfig(6.0)
parts=[('correct','Correct','#4D8073'),('gave_up','Gave up','#C17A61'),
       ('time_limit','Time limit','#757575'),('other_incomplete','Other','#C9C9C9')]
left=np.zeros(len(MODE_ORDER))
for key,label,c in parts:
    values=100*outcome_df[key].to_numpy()/outcome_df['n'].to_numpy()
    ax.barh(MODE_ORDER,values,left=left,height=0.57,label=label,color=c,edgecolor='white',linewidth=0.4)
    left+=values
ax.invert_yaxis()
ax.set_xlim(0,100)
ax.set_xlabel('Trials (%)')
ax.legend(loc='lower center',bbox_to_anchor=(0.5,1.00),ncol=2,frameon=False,
          columnspacing=0.6,handlelength=1.3)
scientific_axes(ax,'x')
fig.subplots_adjust(left=0.18,right=0.98,bottom=0.17,top=0.78)
save_figure(fig,'fig06_trial_outcomes')

# 07 — paired mean differences with participant-level bootstrap CI.
time_matrix=metric_matrix('Tcorrect_analysis_ms')/60000
comparisons=[]
for level in (1,3):
    for m in ('M3','M4','M5'):
        dif=(time_matrix[(m,level)]-time_matrix[('M1',level)]).to_numpy()
        est,lo,hi=bootstrap_stat(dif,'mean')
        comparisons.append(dict(mode=m,level=level,n=len(dif),mean_diff_min=est,ci_low_min=lo,ci_high_min=hi))
paired_df=pd.DataFrame(comparisons)
save_csv(paired_df,'table_paired_differences')
fig,ax=newfig(7.5)
ylabels=[]
for i,item in enumerate(comparisons):
    c=COLORS[item['mode']]
    ax.errorbar(item['mean_diff_min'],i,xerr=[[item['mean_diff_min']-item['ci_low_min']],
                                                [item['ci_high_min']-item['mean_diff_min']]],
                color=c,marker=MARKERS[item['mode']],ls='none',capsize=2,
                capthick=0.9,markersize=4)
    ylabels.append(f"{item['mode']} − M1, C{item['level']}")
ax.set_yticks(range(len(ylabels)),ylabels)
ax.invert_yaxis()
ax.axvline(0,color='black',linewidth=0.7,linestyle='--')
ax.set_xlabel('Mean paired difference (min)')
scientific_axes(ax,'x')
fig.subplots_adjust(left=0.40,right=0.98,bottom=0.15,top=0.97)
save_figure(fig,'fig07_paired_differences')

# 08 — Difference-in-differences: participants are independent units.
def paired_signflip_p(difference,permutations=PERMUTATIONS):
    v=np.asarray(difference,dtype=float)
    obs=abs(float(np.mean(v)))
    # Two-sided permutation test based on sign-flips of paired differences.
    total=0
    chunk=1000
    for start in range(0,permutations,chunk):
        n=min(chunk,permutations-start)
        signs=rng.integers(0,2,size=(n,len(v)))*2-1
        draws=np.abs(np.mean(signs*v,axis=1))
        total+=int(np.sum(draws>=obs-1e-12))
    return (total+1)/(permutations+1)

def holm_adjust(p):
    p=np.asarray(p,float)
    order=np.argsort(p)
    result=np.zeros(len(p))
    current=0.0
    for rank,idx in enumerate(order):
        current=max(current,(len(p)-rank)*p[idx])
        result[idx]=min(current,1.0)
    return result

interaction_rows=[]
for m in ('M3','M4','M5'):
    within=(time_matrix[(m,3)]-time_matrix[('M1',3)]-time_matrix[(m,1)]+time_matrix[('M1',1)]).to_numpy()
    est,lo,hi=bootstrap_stat(within,'mean')
    p=paired_signflip_p(within)
    interaction_rows.append(dict(mode=m,n=len(within),mean_interaction_min=est,
                                 ci_low_min=lo,ci_high_min=hi,p_perm_unadjusted=p))
interaction_df=pd.DataFrame(interaction_rows)
interaction_df['p_holm']=holm_adjust(interaction_df['p_perm_unadjusted'])
save_csv(interaction_df,'table_interaction_contrasts')
fig,ax=newfig(5.5)
for i,row in interaction_df.iterrows():
    ax.errorbar(row['mean_interaction_min'],i,
        xerr=[[row['mean_interaction_min']-row['ci_low_min']],
              [row['ci_high_min']-row['mean_interaction_min']]],
        color=COLORS[row['mode']],marker=MARKERS[row['mode']],ls='none',
        capsize=2.5,markersize=5)
ax.axvline(0,color='black',ls='--',lw=0.75)
ax.set_yticks(range(3),interaction_df['mode'])
ax.invert_yaxis()
ax.set_xlabel('Interaction vs. M1 (min)')
scientific_axes(ax,'x')
fig.subplots_adjust(left=0.20,right=0.97,bottom=0.22,top=0.97)
save_figure(fig,'fig08_interaction_contrast')
print(interaction_df[['mode','mean_interaction_min','ci_low_min','ci_high_min','p_holm']].to_string(index=False))

## 6. System latency: LLM, ASR, and autonomous agent

**Implementation note:** M4 speech recognition uses the **RouterAI API and `nvidia/nemotron-3.5-asr-streaming-multilingual-0.6b`**, not locally hosted GigaAM. For each latency component, we first calculate a median within each participant, then the median of those participant values and a 95% participant-bootstrap interval. Multiple API calls are **not** treated as independent participants. M5 `Tagent_ms` is end-to-end agent execution time rather than only LLM network latency. Missing measurements are not replaced by zeros.


In [ ]:
selected_ids=set(trials['trial_id'])
source_tables=[
    ('M3 LLM', interpretations,'M3','Tllm_ms'),
    ('M4 ASR',transcriptions,'M4','Tasr_ms'),
    ('M4 LLM',interpretations,'M4','Tllm_ms'),
    ('M5 Agent',agent_runs,'M5','Tagent_ms')
]
latency_rows=[]
latency_subject=[]
for label,src,mode,column in source_tables:
    df=src.loc[(src['mode']==mode)&src['trial_id'].isin(selected_ids),
               ['participant_code','trial_id',column]].copy()
    df[column]=pd.to_numeric(df[column],errors='coerce')/1000
    df=df.loc[df[column].notna() & (df[column]>=0)]
    per_sub=df.groupby('participant_code')[column].median()
    if len(per_sub)==0:
        raise ValueError('Latency data are missing for '+label)
    est,lo,hi=bootstrap_stat(per_sub.values,'median')
    latency_rows.append(dict(stage=label,n_requests=len(df),n_participants=len(per_sub),
                             participant_median_s=est,ci_low_s=lo,ci_high_s=hi,
                             request_p90_s=float(df[column].quantile(.9))))
    latency_subject.append(per_sub)
latency_df=pd.DataFrame(latency_rows)
save_csv(latency_df,'table_system_latencies')
fig,ax=newfig(5.8)
for i,row in latency_df.iterrows():
    color=[COLORS['M3'],COLORS['M4'],COLORS['M2'],COLORS['M5']][i]
    ax.errorbar(row['participant_median_s'],i,
                xerr=[[row['participant_median_s']-row['ci_low_s']],
                      [row['ci_high_s']-row['participant_median_s']]],
                color=color,marker='o',ms=4,ls='none',capsize=2,elinewidth=1)
ax.set_yticks(range(len(latency_df)),latency_df['stage'])
ax.invert_yaxis()
ax.set_xlabel('Latency (s)')
scientific_axes(ax,'x')
fig.subplots_adjust(left=0.32,right=0.98,bottom=0.19,top=0.97)
save_figure(fig,'fig09_system_latencies')
print(latency_df[['stage','n_requests','n_participants','participant_median_s']].to_string(index=False))

## 7. Task Mining: interface events and observed system waits

`input`/`action` represent recorded UI events, not direct measurements of cognitive workload. Clicks, keydown events, and scrolls have different meanings and **must not be added together as a single workload score**. System wait time is calculated by taking the union of `request_started` intervals described by `offset_ms` and `duration_ms`, clipped to the observed trial duration. It measures **recorded request waiting**, not the entire difference between completion time and active time. Event-log completeness must be checked, particularly for unsuccessful trials.


In [ ]:
require_columns(events,['trial_id','event_kind','action','offset_ms','duration_ms'], 'events')
e=events.loc[events['trial_id'].isin(selected_ids)].copy()
for col in ('offset_ms','duration_ms'):
    e[col]=pd.to_numeric(e[col],errors='coerce')
action_names=['click','keydown','scroll','pointermove','change']
actions=(e.loc[e['event_kind']=='input']
         .groupby(['trial_id','action']).size().unstack(fill_value=0)
         .reindex(columns=action_names,fill_value=0))
actions=actions.reindex(trials.trial_id,fill_value=0)
actions.index.name='trial_id'
trials=trials.merge(actions.reset_index(),on='trial_id',validate='one_to_one',how='left')

# Merge overlapping wait intervals and clip them to the trial boundary.
duration_by_trial=trials.set_index('trial_id')['elapsed_ms'].to_dict()
def union_request_ms(group):
    cap=float(duration_by_trial.get(group.name,np.nan))
    if not np.isfinite(cap): return np.nan
    spans=[]
    for off,duration in group[['offset_ms','duration_ms']].itertuples(index=False,name=None):
        if not np.isfinite(off) or not np.isfinite(duration): continue
        a=max(0.0,float(off));b=min(cap,a+max(0.,float(duration)))
        if b>a: spans.append((a,b))
    spans.sort()
    total=0.; begin=None; end=None
    for a,b in spans:
        if begin is None: begin,end=a,b
        elif a<=end: end=max(end,b)
        else: total+=end-begin; begin,end=a,b
    if begin is not None: total+=end-begin
    return total

request_ev=e.loc[e['event_kind']=='request_started']
wait_by_trial=request_ev.groupby('trial_id').apply(union_request_ms,include_groups=False)
trials['request_wait_ms']=trials['trial_id'].map(wait_by_trial).fillna(0.0)
trials['request_wait_pct']=np.where(trials['elapsed_ms']>0,
                                   100*trials['request_wait_ms']/trials['elapsed_ms'],np.nan)
if (trials['request_wait_pct']>100.001).any():
    raise ValueError('Invalid request-wait fraction: exceeds 100%')
if trials['request_wait_pct'].isna().any():
    raise ValueError('Missing elapsed_ms; request-wait fraction cannot be computed')

# 10–11 — UI-event counts per trial (median with participant-bootstrap CI).
curve_chart('click','fig10_clicks_by_complexity','Click events / trial')
curve_chart('keydown','fig11_keydown_by_complexity','Keydown events / trial')
# 12 — Fraction of observed elapsed time occupied by recorded system requests.
curve_chart('request_wait_pct','fig12_request_wait_share',
            'Request waiting (% of elapsed)',ylim=(0,100))

mining_rows=[]
for m in MODE_ORDER:
    q=trials.loc[trials['mode']==m]
    mining_rows.append(dict(mode=m,n_trials=len(q),n_trials_no_events=int((~q['trial_id'].isin(e.trial_id)).sum()),
        median_clicks=float(q['click'].median()),median_keydown=float(q['keydown'].median()),
        median_scroll=float(q['scroll'].median()),median_pointermove=float(q['pointermove'].median()),
        median_wait_s=float(q['request_wait_ms'].median()/1000),
        median_wait_percent=float(q['request_wait_pct'].median())))
mining_df=pd.DataFrame(mining_rows)
save_csv(mining_df,'table_task_mining')
print(mining_df[['mode','median_clicks','median_keydown','median_scroll','median_wait_percent']].to_string(index=False))

## 8. Journal-ready tables (8 cm PNG and accompanying CSV)

The results table is split into **three panels by complexity level**, each containing five modes. This is more legible in a narrow journal column than a 15-row table with tiny text. Timing summaries are descriptive: unsuccessful trials receive the protocol-defined **analysis-time penalty**, while `Actual time` is computed **only for successful trials**. Numerical values are also exported as CSV.


In [ ]:
rows=[]
for (m,l),q in trials.groupby(['mode','level'],observed=True):
    success=q.loc[q['success']==1]
    rows.append(dict(mode=str(m),complexity=f'C{l}',n=len(q),success_n=int(q['success'].sum()),
        success_pct=100*float(q['success'].mean()),A1_analysis_pct=100*float(q['A1'].mean()),
        A1_observed_n=int(q['A1_actual'].notna().sum()),
        time_analysis_median_s=float(q['Tcorrect_analysis_ms'].median()/1000),
        time_analysis_mean_s=float(q['Tcorrect_analysis_ms'].mean()/1000),
        time_actual_success_median_s=float(success['Tcorrect_actual_ms'].median()/1000),
        user_active_median_s=float(q['Tuser_active_ms'].median()/1000),
        retry_analysis_mean=float(q['Nretry_analysis'].mean()),
        retry_actual_mean=float(q['Nretry_actual'].mean())))
by_cell_df=pd.DataFrame(rows).sort_values(['complexity','mode'])
save_csv(by_cell_df,'table_mode_by_complexity_full')

by_mode_df=(trials.groupby('mode',observed=True).agg(
    n=('trial_id','count'),successful=('success','sum'),
    success_pct=('success',lambda v:100*v.mean()),
    A1_pct=('A1',lambda v:100*v.mean()),
    time_penalty_median_s=('Tcorrect_analysis_ms',lambda v:v.median()/1000),
    user_active_median_s=('Tuser_active_ms',lambda v:v.median()/1000),
    mean_retries=('Nretry_analysis','mean')).reset_index())
save_csv(by_mode_df,'table_mode_overall_full')
save_csv(flow_table,'table_population_flow')

def table_png(values,headers,stem,widths=None,notes=None):
    data=[[str(item) for item in row] for row in values]
    rows_count=len(data)
    height_cm=1.5+0.61*(rows_count+1)+(0.65 if notes else 0)
    fig,ax=newfig(height_cm)
    ax.axis('off')
    if widths is None: widths=[1/len(headers)]*len(headers)
    table=ax.table(cellText=data,colLabels=headers,loc='center',
                   cellLoc='center',colLoc='center',colWidths=widths,
                   bbox=[0.025,0.13 if notes else 0.05,0.95,0.75 if notes else 0.88])
    table.auto_set_font_size(False)
    table.set_fontsize(6.35)
    for (r,c),cell in table.get_celld().items():
        cell.set_linewidth(0)
        cell.set_edgecolor('#4c4c4c')
        if r==0:
            cell.set_facecolor('#e8e8e8')
            cell.set_text_props(weight='bold')
        elif r%2==0:cell.set_facecolor('#f7f7f7')
    if notes: ax.text(.025,.01,notes,transform=ax.transAxes,va='bottom',ha='left',fontsize=5.7)
    path=save_figure(fig,stem)
    target=TABLE_DIR/path.name
    path.replace(target)
    figure_paths.remove(path)
    table_paths.append(target)
    return target

for level in LEVEL_ORDER:
    ss=by_cell_df[by_cell_df['complexity']==f'C{level}']
    entries=[]
    for _,row in ss.iterrows():
        entries.append([row['mode'],f"{int(row['success_n'])}/{N_SUBJECTS}",
                        f"{row['A1_analysis_pct']:.0f}",f"{row['time_analysis_median_s']:.0f}",
                        f"{row['user_active_median_s']:.0f}",
                        f"{row['retry_analysis_mean']:.1f}"])
    table_png(entries,['Mode','Succ.','A1 %','T corr., s','Active, s','Retries'],
              f'table_c{level}_summary',widths=[.13,.16,.13,.23,.2,.15],
              notes=f'C{level}; N={N_SUBJECTS}; T corr. includes protocol penalty.')

# Additional compact latency table
entries=[[r['stage'],str(int(r['n_requests'])),f"{r['participant_median_s']:.2f}",
          f"{r['request_p90_s']:.1f}"] for _,r in latency_df.iterrows()]
table_png(entries,['Stage','Calls','Median, s','P90, s'],'table_latency_summary',
          widths=[.39,.15,.23,.23],notes='Median across participant medians; P90 across calls.')
print('PNG figures:',len(figure_paths), 'PNG tables:',sum(p.suffix=='.png' for p in table_paths), 'CSV:',sum(p.suffix=='.csv' for p in table_paths))

## 9. Quality control, Methods reporting, and reproducibility

`analysis_manifest.json` records the ZIP SHA-256 digest, the dual inclusion rule (`P###` and `complete=True`), the generated artifacts, and important interpretation limits. **Published CSV summaries are aggregated** by mode and complexity rather than exporting participant transcripts, audio content, coordinates, or personal identifiers. The paper should explicitly state that (a) `Tcorrect_analysis_ms` is the prespecified penalty-inclusive endpoint, (b) completion is a **session-level** status, (c) complexity levels appeared in the fixed order C1 → C2 → C3, and (d) logged system waits cannot account for all inactive time.


## 10. Multivariate bubble charts (C1–C3)

**Each bubble represents one interaction mode (N = 37 participants per complexity level in this dataset).** The horizontal axis shows **mean protocol-penalized analysis time**, and the vertical axis shows **first-attempt accuracy**. Bubble **area** is proportional to mean user-active time. Mode is encoded by saturated color and a **black external label connected to its bubble**. Area—not radius—is scaled linearly using a **common scale across C1–C3**. The time axis uses a logarithmic scale to accommodate widely differing durations. These plots are **descriptive summaries**; neither the 555 trials nor the plotted bubbles should be interpreted as independent experimental subjects.


In [ ]:
# 13–15: bubble charts, separate complexity levels for true 8 cm publication width.
bubble_df=(trials.groupby(['mode','level'],observed=True)
           .agg(time_penalty_mean_s=('Tcorrect_analysis_ms',lambda x:x.mean()/1000),
                first_correct_pct=('A1','mean'),
                active_mean_s=('Tuser_active_ms',lambda x:x.mean()/1000),
                n=('trial_id','size')).reset_index())
bubble_df['first_correct_pct']*=100
save_csv(bubble_df,'table_bubble_mode_level')
size_floor,size_ceiling=75.,830.
all_active=bubble_df['active_mean_s'].to_numpy()
size_min,size_max=float(np.min(all_active)),float(np.max(all_active))
def bubble_area(s):
    return size_floor+(float(s)-size_min)/(size_max-size_min+1e-9)*(size_ceiling-size_floor)

# Offsets are in display points. Keeping labels outside the colored marks
# preserves their readability in a single-column 8 cm printed figure.
BUBBLE_LABEL_OFFSETS = {
    1: {'M1':(16,-11),'M2':(-18,-13),'M3':(20,-13),'M4':(12,12),'M5':(14,12)},
    2: {'M1':(-22,-12),'M2':(21,12),'M3':(-18,14),'M4':(15,12),'M5':(16,-14)},
    3: {'M1':(-20,-14),'M2':(20,15),'M3':(20,14),'M4':(-16,14),'M5':(18,-14)},
}
for lev in LEVEL_ORDER:
    s=bubble_df[bubble_df.level==lev]
    fig,ax=newfig(7.1)
    for _,r in s.iterrows():
        m=str(r['mode']); x=float(r['time_penalty_mean_s']);y=float(r['first_correct_pct'])
        ax.scatter(x,y,s=bubble_area(r['active_mean_s']),facecolor=COLORS[m],
                   edgecolor='#202020',linewidth=0.6,alpha=.91,zorder=4)
        dx,dy=BUBBLE_LABEL_OFFSETS[lev][m]
        ax.annotate(m,(x,y),xytext=(dx,dy),textcoords='offset points',
                    va='center',ha='center',fontsize=7.1,fontweight='bold',color='#111111',
                    bbox=dict(boxstyle='round,pad=.12',facecolor='white',edgecolor='none',alpha=.95),
                    arrowprops=dict(arrowstyle='-',lw=.55,color='#303030',shrinkA=2,shrinkB=2),
                    zorder=7)
    ax.set_xscale('log')
    ax.set_xlim(max(10,bubble_df.time_penalty_mean_s.min()*.70),
                 bubble_df.time_penalty_mean_s.max()*1.45)
    ax.set_ylim(max(0,s.first_correct_pct.min()-14),105)
    ax.set_ylabel('First-attempt accuracy (%)')
    ax.set_xlabel('Mean time incl. penalty (s, log)')
    ax.set_title(f'C{lev}: time / accuracy / user activity',fontsize=7.3)
    scientific_axes(ax)
    # Legend bubble area is globally comparable between all three figures.
    fig.text(.22,.013,'Bubble AREA = mean active user time (s)',ha='left',fontsize=5.65)
    fig.subplots_adjust(left=.21,right=.96,top=.89,bottom=.25)
    save_figure(fig,f'fig{12+lev:02d}_bubble_c{lev}')
print('Bubble plots: three complexity levels, equal area encoding across panels.')

## 11. Cumulative success over *observed* time

For every M1–M5 × C1–C3 cell, the graph reports the **empirical cumulative proportion of all assigned trials** that achieved a correct answer by time *t*. The event time is `Tcorrect_actual_ms`, **only for `status=correct`**. Unsuccessful trials remain unsuccessful; they never become artificial successes at the penalty threshold. The time axis is displayed up to **700 s for C1** and **1200 s for C2–C3** (horizontal axis), while the calculations retain the full observation window. Each curve levels off at the observed final success rate, which may be less than 100%. This is an **empirical cumulative proportion**, not a Kaplan–Meier estimator; unsuccessful outcomes are not assumed to be independent right-censoring.


In [ ]:
# 16–18: cumulative success (empirical subdistribution), failures stay failures.
thresholds_s=np.array([30,60,120,180,300,600,900,1200,1500],dtype=int)
success_by_time_rows=[]
for lev in LEVEL_ORDER:
    fig,ax=newfig(6.5)
    for m in MODE_ORDER:
        q=trials.loc[(trials['mode']==m)&(trials['level']==lev)]
        times=(q.loc[q.success==1,'Tcorrect_actual_ms'].to_numpy(float)/1000)
        times=np.sort(times[np.isfinite(times)])
        horizon=float(q.trial_limit_seconds.max()) if 'trial_limit_seconds' in q else 1500.
        x=np.r_[0,times,horizon]
        y=100*np.r_[0,np.arange(1,len(times)+1),len(times)]/len(q)
        ax.step(x,y,where='post',color=COLORS[m],linestyle=LINE_STYLES[m],
                label=m,linewidth=1.2)
        for t in thresholds_s:
            success_by_time_rows.append(dict(mode=m,level=lev,time_s=int(t),
                successes=int(np.sum(times<=t)),n=len(q),success_pct=float(100*np.sum(times<=t)/len(q))))
    ax.set_xlim(0,700 if lev == 1 else 1200); ax.set_ylim(0,103)
    ax.set_xticks(np.arange(0,701 if lev==1 else 1201,100 if lev==1 else 200))
    ax.set_xlabel('Elapsed time to correct answer (s)')
    ax.set_ylabel('Cumulative success (%)')
    ax.set_title(f'C{lev}: correctly solved by time t',fontsize=7.6)
    ax.legend(ncol=5,loc='lower right',frameon=False,columnspacing=.7,handlelength=1.3)
    scientific_axes(ax)
    fig.subplots_adjust(left=.20,right=.97,top=.90,bottom=.19)
    save_figure(fig,f'fig{15+lev:02d}_cumulative_success_c{lev}')
success_time_df=pd.DataFrame(success_by_time_rows)
save_csv(success_time_df,'table_cumulative_success_at_times')

## 12. Paired participant differences and raincloud distributions

For C3, the paired effect plot shows **37 individual within-participant** protocol-penalized time differences from M1, alongside the estimated mean contrast and a 95% **participant-cluster bootstrap confidence interval**. The C1–C3 raincloud figures display the **full distribution of penalized time**, including unsuccessful trials. The half-violin is a density estimate; the thick segment indicates the interquartile range, the dot marks the median, and translucent observations represent individual trials. Density is descriptive, especially when multiple trials have exactly 1500 s penalty values.


In [ ]:
# 19: paired effect plot with observed participant differences, bootstrap mean 95% CI.
fig,ax=newfig(7.0)
paired_raw=[]
for i,m in enumerate(('M3','M4','M5')):
    x=(time_matrix[(m,3)]-time_matrix[('M1',3)]).to_numpy(float)
    jit=rng.uniform(-.12,.12,len(x))
    ax.scatter(x,np.full(len(x),i)+jit,s=9,color=COLORS[m],alpha=.54,lw=0,zorder=2)
    est,lo,hi=bootstrap_stat(x,'mean')
    ax.errorbar(est,i,xerr=[[est-lo],[hi-est]],color='black',marker='D',mfc='white',
                ms=4.4,lw=1.5,capsize=2.5,zorder=5)
    for student,v in zip(subjects,x):
        paired_raw.append(dict(participant_code=student,mode=m,level=3,diff_vs_M1_min=float(v)))
ax.axvline(0,ls='--',color='#888888',lw=.7)
ax.set_yticks(range(3),['M3 − M1','M4 − M1','M5 − M1']);ax.invert_yaxis()
ax.set_xlabel('Participant paired difference in C3 (min)')
ax.set_title('C3: participant-level paired effects',fontsize=7.5)
scientific_axes(ax,'x')
fig.text(.22,.02,'Hollow diamond: mean; bar: bootstrap 95% CI',fontsize=5.6)
fig.subplots_adjust(left=.25,right=.97,top=.89,bottom=.27)
save_figure(fig,'fig19_paired_effects_c3_raw')
save_csv(pd.DataFrame(paired_raw),'table_paired_subject_differences_c3')

# 20–22: half-violin density, individual observations and box summaries.
def kde_gaussian(values,xs):
    v=np.asarray(values,float)
    v=v[np.isfinite(v)]
    sd=float(np.std(v,ddof=1)) if len(v)>1 else 0
    spread=float(np.subtract(*np.percentile(v,[75,25])))/1.349 if len(v)>1 else 0
    robust_sd=min(sd,spread) if sd>0 and spread>0 else max(sd,spread,1.)
    bandwidth=max(12.,.9*robust_sd*len(v)**(-.2))
    return np.exp(-.5*((xs[:,None]-v[None,:])/bandwidth)**2).mean(axis=1)/bandwidth

for lev in LEVEL_ORDER:
    fig,ax=newfig(7.4)
    for k,m in enumerate(MODE_ORDER):
        ss=trials.loc[(trials['mode']==m)&(trials['level']==lev),'Tcorrect_analysis_ms'].to_numpy(float)/1000
        hi=max(25.,float(np.max(ss)))
        xs=np.linspace(0,hi+max(30.,hi*.08),180)
        density=kde_gaussian(ss,xs)
        density=density/max(density.max(),1e-12)*.32
        ax.fill_between(xs,k,k+density,color=COLORS[m],alpha=.50,lw=0)
        ax.plot(xs,k+density,color=COLORS[m],lw=.65)
        jit=-.16-rng.random(len(ss))*.17
        ax.scatter(ss,k+jit,s=5.1,alpha=.55,color=COLORS[m],lw=0)
        q1,med,q3=np.percentile(ss,[25,50,75])
        ax.plot([q1,q3],[k-.05,k-.05],color=COLORS[m],lw=3.3,solid_capstyle='butt')
        ax.plot(med,k-.05,marker='o',color='black',ms=2.3)
    ax.set_yticks(np.arange(5),MODE_ORDER); ax.invert_yaxis()
    ax.set_ylim(4.65,-.60)
    ax.set_xlabel('Analysis time incl. penalty (s)')
    ax.set_title(f'C{lev}: participant distributions (n=37)',fontsize=7.5)
    scientific_axes(ax,'x')
    fig.subplots_adjust(left=.18,right=.97,top=.89,bottom=.16)
    save_figure(fig,f'fig{19+lev:02d}_raincloud_c{lev}')

## 13. Outcome categories and decomposition of observed trial time

100% stacked bars separate outcome categories for **each** mode and complexity level; trials with no submitted attempts are not counted as successes. Time decomposition uses **observed** `elapsed_ms` rather than penalized `Tcorrect_analysis_ms`. Components are estimated user-active time (`Tuser_active_ms`), the **union** of logged system request intervals (`request_wait_ms`), and the residual (`elapsed − active − requests`). The residual is **not directly observed thinking time**: it may reflect pauses, unlogged activity, loss of focus, and other gaps. Components are calculated **within each trial**, then averaged at participant level. If an impossible negative residual is detected, execution stops rather than silently changing the data.


In [ ]:
# 23–25: conditional outcomes by mode and complexity.
outcome_by_cell=[]
for lev in LEVEL_ORDER:
    fig,ax=newfig(6.1)
    left=np.zeros(5)
    for key,label,color in parts:
        proportions=[]
        for m in MODE_ORDER:
            q=trials.loc[(trials['mode']==m)&(trials['level']==lev)]
            if key=='correct': count=int(q.success.sum())
            elif key=='gave_up': count=int(q.gaveup_flag.sum())
            elif key=='time_limit': count=int(q.timeout_flag.sum())
            else: count=int((~((q.success==1)|q.gaveup_flag|q.timeout_flag)).sum())
            proportions.append(100*count/len(q))
            outcome_by_cell.append(dict(mode=m,level=lev,outcome=key,n=count,total=len(q),percent=100*count/len(q)))
        ax.barh(MODE_ORDER,proportions,left=left,color=color,height=.64,
                label=label,edgecolor='white',linewidth=.5)
        left+=np.asarray(proportions)
    ax.invert_yaxis();ax.set_xlim(0,100);ax.set_xlabel('Trial outcomes (%)')
    fig.legend(loc='upper center',bbox_to_anchor=(.5,.98),ncol=2,frameon=False,
               handlelength=1.0,columnspacing=.65)
    scientific_axes(ax,'x')
    fig.subplots_adjust(left=.18,right=.97,bottom=.18,top=.76)
    save_figure(fig,f'fig{22+lev:02d}_outcomes_c{lev}')
save_csv(pd.DataFrame(outcome_by_cell),'table_outcomes_mode_complexity')

# 26–28: decomposition of observed (NOT penalty-imputed) elapsed time.
trials['time_unattributed_ms']=(trials['elapsed_ms']-trials['Tuser_active_ms']-
                                 trials['request_wait_ms'])
if (trials.time_unattributed_ms < -1).any():
    raise ValueError('Tuser_active + request_wait > elapsed_ms: cannot create non-overlapping stacked budget')
trials['time_unattributed_ms']=trials['time_unattributed_ms'].clip(lower=0)
components=[('Tuser_active_ms','Active user','#4B7BA3'),
            ('request_wait_ms','System requests','#DAA748'),
            ('time_unattributed_ms','Other / unobserved','#999999')]
time_budget_rows=[]
for lev in LEVEL_ORDER:
    fig,ax=newfig(6.5)
    left=np.zeros(5)
    for field,label,color in components:
        vals=[]
        for m in MODE_ORDER:
            q=trials[(trials['mode']==m)&(trials['level']==lev)]
            amount=float(q[field].mean()/1000)
            vals.append(amount)
            time_budget_rows.append(dict(mode=m,level=lev,component=field,mean_s=amount,n=len(q)))
        ax.barh(MODE_ORDER,vals,left=left,height=.63,color=color,edgecolor='white',lw=.3,label=label)
        left+=np.asarray(vals)
    ax.invert_yaxis();ax.set_xlabel('Mean observed elapsed time (s)')
    fig.legend(loc='upper center',bbox_to_anchor=(.5,.98),frameon=False,ncol=2,
               handlelength=1.1,columnspacing=.7)
    scientific_axes(ax,'x')
    fig.subplots_adjust(left=.18,right=.97,bottom=.18,top=.76)
    save_figure(fig,f'fig{25+lev:02d}_time_budget_c{lev}')
save_csv(pd.DataFrame(time_budget_rows),'table_time_budget_mode_complexity')

## 14. Task Mining: retries, user actions, and agent tools

**Retry turnaround** is measured only between completion of an unsuccessful result check and the beginning of the **next submitted attempt in the same trial**. If no further attempt is made, no interval is invented. This duration is not pure editing time; it may include reviewing feedback and deciding how to proceed.

**M5 agent:** `agent_runs.trajectory` records observable `tool` and `llm` steps. The tool heatmap uses the seven observed tool names. LLM and tool calls are summed at **trial level** (including repeat attempts) before comparing complexity levels. Technical tool-call duration is not equal to total user task time. Hidden model reasoning is not reconstructed.


In [ ]:
# 29: repair/retry turnaround after a failed submission.
require_columns(attempts,['trial_id','ordinal','correct','started_ms','finished_ms'],'attempts')
a=attempts.loc[attempts.trial_id.isin(selected_ids),['trial_id','ordinal','correct','started_ms','finished_ms']].copy()
for col in ['ordinal','started_ms','finished_ms']:
    a[col]=pd.to_numeric(a[col],errors='coerce')
a['correct']=to_bool(a['correct'])
a=a.sort_values(['trial_id','ordinal'])
a['next_started_ms']=a.groupby('trial_id')['started_ms'].shift(-1)
a['repair_sec']=(a['next_started_ms']-a['finished_ms'])/1000
repair=a.loc[(~a.correct)&a.next_started_ms.notna()].copy()
if (repair.repair_sec < 0).any():
    raise ValueError('Negative interval between failed verification and next submission')
repair=repair.merge(trials[['trial_id','participant_code','mode','level']],on='trial_id',validate='many_to_one')
save_csv(repair[['trial_id','participant_code','mode','level','ordinal','repair_sec']],
         'table_retry_repair_intervals')
# Every mode's interval distribution (individual intervals are NOT independent subjects).
fig,ax=newfig(6.8)
repair_medians=[]
for i,m in enumerate(MODE_ORDER):
    ss=repair.loc[repair['mode']==m,'repair_sec'].to_numpy(float)
    if len(ss):
        # Robust display in log1p seconds; mark trial intervals individually, subject-level summary in CSV.
        xx=np.log1p(ss)
        ax.scatter(xx,np.full(len(ss),i)+rng.uniform(-.19,.19,len(ss)),s=7,
                   color=COLORS[m],alpha=.56,lw=0)
        med=np.median(ss)
        ax.plot(np.log1p(med),i,marker='D',color='black',ms=4,zorder=4)
        repair_medians.append(dict(mode=m,n_intervals=len(ss),median_s=float(med)))
    else:
        repair_medians.append(dict(mode=m,n_intervals=0,median_s=np.nan))
ax.set_yticks(range(5),MODE_ORDER);ax.invert_yaxis()
xticks=[0,10,30,60,120,300,600]
ax.set_xticks(np.log1p(xticks),[str(t) for t in xticks]);ax.set_xlim(-.1,np.log1p(850))
ax.set_xlabel('Time until next submission (s, log1p)')
ax.set_title('Retry correction / reflection delay',fontsize=7.3)
scientific_axes(ax,'x')
fig.subplots_adjust(left=.19,right=.97,bottom=.21,top=.90)
save_figure(fig,'fig29_retry_turnaround')
save_csv(pd.DataFrame(repair_medians),'table_retry_turnaround_summary')

# 30: meaningful UI event types by mode/complexity (not pointermoves)
action_cell=[]
for (m,l),q in trials.groupby(['mode','level'],observed=True):
    for action in ('click','keydown','scroll','change'):
        action_cell.append(dict(mode=str(m),level=int(l),action=action,
                         median_events=float(q[action].median()),mean_events=float(q[action].mean()),n=len(q)))
action_summary=pd.DataFrame(action_cell)
save_csv(action_summary,'table_ui_events_by_mode_level')
fig,ax=newfig(7.6)
# log1p of mean click+keydown: defined exactly and reported in the legend.
heat=np.asarray([[np.log1p(trials.loc[(trials['mode']==m)&(trials['level']==l),
                ['click','keydown']].sum(axis=1).mean()) for l in LEVEL_ORDER] for m in MODE_ORDER])
im=ax.imshow(heat,cmap='YlGnBu',aspect='auto',origin='upper',vmin=0)
ax.set_xticks(range(3),['C1','C2','C3']);ax.set_yticks(range(5),MODE_ORDER)
ax.tick_params(length=0)
for i,m in enumerate(MODE_ORDER):
    for j,lev in enumerate(LEVEL_ORDER):
        mean_n=float(trials.loc[(trials['mode']==m)&(trials['level']==lev),
            ['click','keydown']].sum(axis=1).mean())
        ax.text(j,i,f'{mean_n:.0f}',ha='center',va='center',fontsize=8,
                color=('white' if (lambda c:0.2126*c[0]+0.7152*c[1]+0.0722*c[2])(im.cmap(im.norm(heat[i,j]))) < .48 else '#111111'),fontweight='bold')
ax.set_xlabel('Task complexity')
ax.set_title('Mean click + keydown events / trial',fontsize=7.5)
fig.subplots_adjust(left=.20,right=.96,bottom=.17,top=.90)
save_figure(fig,'fig30_mean_clicks_and_keys_heatmap')

# Actual M5 tool trajectory. Do not infer tool names from a prompt or guessed sequence.
require_columns(agent_runs,['trial_id','mode','trajectory','tool_calls','llm_calls','Tllm_ms','Ttool_ms'],'agent_runs')
agent=agent_runs.loc[agent_runs.trial_id.isin(selected_ids)&(agent_runs['mode']=='M5')].copy()
if agent.empty: raise ValueError('No M5 agent runs among included trials')
import collections
TOOLS=['search_tasks','filter_by_field','group_by','aggregate',
       'find_extremum','sort','export_result']
records=[]
for row in agent.itertuples(index=False):
    if pd.isna(row.trajectory):
        raise ValueError('Agent trajectory is missing')
    trajectory=json.loads(row.trajectory)
    if not isinstance(trajectory,list):raise ValueError('Invalid agent trajectory')
    seen=collections.Counter()
    for action in trajectory:
        if action.get('type')=='tool':
            tool=action.get('tool')
            if tool not in TOOLS:raise ValueError('Unknown agent tool in trajectory: '+str(tool))
            seen[tool]+=1
    for tool in TOOLS:
        records.append(dict(trial_id=row.trial_id,tool=tool,count=seen[tool]))
agent_tools=pd.DataFrame(records).merge(trials[['trial_id','participant_code','level']],
                                    on='trial_id',validate='many_to_one')
# one line per trial AND tool, aggregated across all M5 attempts
all_tool_trial=(agent_tools.groupby(['trial_id','participant_code','level','tool'],observed=True)['count']
                .sum().reset_index())
save_csv(all_tool_trial,'table_m5_tool_counts_per_trial')
tool_mean=all_tool_trial.groupby(['tool','level'],observed=True)['count'].mean().unstack('level').reindex(index=TOOLS,columns=LEVEL_ORDER)
fig,ax=newfig(8.5)
img=ax.imshow(tool_mean.to_numpy(float),cmap='YlGnBu',aspect='auto',origin='upper')
ax.set_xticks(range(3),['C1','C2','C3'])
ax.set_yticks(range(len(TOOLS)),TOOLS,fontsize=6.2)
ax.tick_params(length=0)
for i,tool in enumerate(TOOLS):
    for j,lev in enumerate(LEVEL_ORDER):
        val=float(tool_mean.iloc[i,j]); ax.text(j,i,f'{val:.1f}',ha='center',va='center',
                color='white' if (lambda c:0.2126*c[0]+0.7152*c[1]+0.0722*c[2])(img.cmap(img.norm(val))) < .48 else '#111111',fontsize=7)
ax.set_xlabel('Task complexity')
ax.set_title('M5: tool calls / trial (mean)',fontsize=7.5)
fig.subplots_adjust(left=.39,right=.96,top=.89,bottom=.13)
save_figure(fig,'fig31_m5_tool_usage_heatmap')
save_csv(tool_mean.reset_index(),'table_m5_tool_usage_by_complexity')

# 32: LLM calls and tool calls per M5 trial (not per request).
for fld in ('tool_calls','llm_calls','Tllm_ms','Ttool_ms'):
    agent[fld]=pd.to_numeric(agent[fld],errors='coerce')
agent_totals=(agent.groupby('trial_id',as_index=False)[['tool_calls','llm_calls','Tllm_ms','Ttool_ms']]
              .sum(min_count=1).merge(trials[['trial_id','participant_code','level']],on='trial_id',validate='one_to_one'))
if len(agent_totals)!=N_SUBJECTS*3:
    raise ValueError('Missing agent records for included M5 trials')
save_csv(agent_totals,'table_m5_agent_counts_and_latencies_per_trial')
fig,ax=newfig(6.8)
for field,color,label,marker in [('tool_calls',COLORS['M5'],'Tools','o'),
                                 ('llm_calls',COLORS['M2'],'LLM calls','s')]:
    arr=[]
    for lev in LEVEL_ORDER:
        sub=agent_totals.loc[agent_totals.level==lev,field].to_numpy(float)
        arr.append(bootstrap_stat(sub,stat='mean'))
    agg=np.asarray(arr)
    ax.errorbar(LEVEL_ORDER,agg[:,0],yerr=np.vstack([agg[:,0]-agg[:,1],agg[:,2]-agg[:,0]]),
                color=color,marker=marker,capsize=2,lw=1.2,label=label)
ax.set_xticks(LEVEL_ORDER,['C1','C2','C3']);ax.set_xlabel('Task complexity')
ax.set_ylabel('Calls per M5 trial')
ax.set_title('M5: tool and model iterations',fontsize=7.5)
ax.legend(frameon=False,loc='upper left')
scientific_axes(ax)
fig.subplots_adjust(left=.19,right=.96,top=.90,bottom=.17)
save_figure(fig,'fig32_m5_tool_llm_calls')

# 33: M5 time inside agent (LLM vs tools). These are summed per trial, no extra nesting assumed.
fig,ax=newfig(6.0)
llm=[];tools=[]
for lev in LEVEL_ORDER:
    v=agent_totals.loc[agent_totals.level==lev]
    llm.append(float(v.Tllm_ms.mean()/1000))
    tools.append(float(v.Ttool_ms.mean()/1000))
ax.barh(['C1','C2','C3'],llm,color=COLORS['M5'],label='LLM calls',height=.58)
ax.barh(['C1','C2','C3'],tools,left=llm,color=COLORS['M2'],label='Tools',height=.58)
ax.invert_yaxis();ax.set_xlabel('Mean agent component time (s)')
# Caption belongs to paper; no title colliding with the legend at 8 cm.
fig.legend(loc='upper center',bbox_to_anchor=(.5,.95),frameon=False,ncol=2)
scientific_axes(ax,'x')
fig.subplots_adjust(left=.18,right=.97,top=.78,bottom=.18)
save_figure(fig,'fig33_m5_llm_vs_tools_time')
print('Added 21 new figures for multidimensional, success-time, paired, activity and agent analyses.')

## 15. Selecting visualizations for IEEE THMS

`output/figures/` contains **45 publication-sized figures**, while `output/tables/` contains four rendered PNG tables plus CSV summaries. **Do not include all 45 figures in the main manuscript.** Suggested priorities:

1. **H1/H2:** `fig01`, `fig08`, `fig15` (C3 bubble chart), `fig18` (C3 cumulative success), and `fig19` (paired effects).
2. **Reliability:** `fig02`, `fig03`, `fig25` (C3 outcomes), and `fig22` (C3 time distribution).
3. **Interaction mechanisms:** `fig28` (C3 time decomposition), `fig30` (user actions), `fig31` (M5 tools), and `fig32` (LLM/tool steps).
4. **Validity diagnostics:** `fig37` (block-order association), `fig38`–`fig40` (task variants), and `fig41`–`fig43` (success by attempt).

Move additional complexity-specific panels and supplementary analyses to *Supplementary Materials*. Actual cumulative success is not a survival function with every unsuccessful attempt treated as an event. Bubble charts combine **three descriptive means** and are not significance tests.


## 16. Additional validity and mechanism diagnostics (Figs. 34–45)

These diagnostics **complement rather than replace** the primary mode-effect analysis. They use only the completed sessions of participants with `P###` identifiers; the **participant** remains the unit of independent replication.

- **34–36:** Signed paired win margin at C1, C2, and C3: `(number of participants with row faster − number with column faster) / 37 × 100` percentage points. **Positive** means the row mode is faster more often, **negative** means the row mode is slower more often, and zero means balanced wins and losses; ties contribute zero. These are within-participant descriptive contrasts, not causal probabilities. The accompanying CSV also retains the original win-rate definition (ties counted as half).
- **37:** Association of block position and completion time after descriptive adjustment for participant and mode × complexity.
- **38–40:** Variation across nominally isomorphic task variants a–e within each complexity level. Fixed participant, mode, and block-position effects are included; exploratory intervals use participant jackknife estimates.
- **41–43:** Proportion of trials successfully completed on or before attempt *k*, keeping unsuccessful trials in the denominator.
- **44:** M4 speech duration versus external ASR API latency; no speech-recognition WER is inferred without reference transcripts.
- **45:** M5 total agent tool calls versus agent latency, with observed outcomes and complexity levels distinguished.

All plots are primarily descriptive or exploratory. Complexity is presented in a fixed order (C1 → C2 → C3) within each block; statistical adjustment cannot fully remove this confounding. Each chart is exported **as PNG only**, 8 cm wide at 300 dpi.


In [ ]:
# 34–36. Signed within-participant win margin (percentage points), including penalties.
# A tie contributes to neither wins nor losses. The sign distinguishes
# faster (+) from slower (-); unlike a raw win rate, values can be negative.
from collections import defaultdict
from math import sqrt

win_rows=[]
wide=metric_matrix('Tcorrect_analysis_ms')
for lev in LEVEL_ORDER:
    margin=np.full((len(MODE_ORDER),len(MODE_ORDER)),np.nan)
    for i,mi in enumerate(MODE_ORDER):
        for j,mj in enumerate(MODE_ORDER):
            if i==j: continue
            diff=wide[(mi,lev)].to_numpy(float)-wide[(mj,lev)].to_numpy(float)
            wins=int(np.sum(diff<0)); ties=int(np.sum(diff==0)); losses=int(np.sum(diff>0))
            win_rate=(wins+0.5*ties)/len(diff)
            advantage_pp=100*(wins-losses)/len(diff)
            margin[i,j]=advantage_pp
            win_rows.append(dict(level=lev,row_mode=mi,column_mode=mj,
                        participants=len(diff),wins=wins,ties=ties,losses=losses,
                        row_faster_pct=100*wins/len(diff),
                        column_faster_pct=100*losses/len(diff),
                        ties_pct=100*ties/len(diff),
                        signed_win_margin_pp=advantage_pp,
                        proportion_faster_with_half_ties=win_rate,
                        outcome='Tcorrect_analysis_ms; positive margin means row faster'))
    fig,ax=newfig(7.35)
    # A diverging color scale uses 0 as the neutral, equal-win point.
    im=ax.imshow(np.ma.masked_invalid(margin),cmap='RdBu',vmin=-100,vmax=100,
                 aspect='equal',interpolation='nearest')
    ax.set_xticks(range(5),MODE_ORDER);ax.set_yticks(range(5),MODE_ORDER)
    ax.set_xlabel('Comparison mode (column)')
    ax.set_ylabel('Candidate mode (row)')
    ax.tick_params(length=0)
    for i in range(5):
        for j in range(5):
            if i==j:
                label='—'; color='#111111'
            else:
                label='0' if abs(margin[i,j])<0.5 else f'{margin[i,j]:+0.0f}'
                color='white' if abs(margin[i,j])>62 else '#111111'
            ax.text(j,i,label,ha='center',va='center',fontsize=7.0,color=color,
                    fontweight='bold' if i!=j else 'normal')
    fig.text(.51,.051,'Win margin (percentage points): + row faster / − row slower',
             fontsize=5.1,ha='center',color='#111111')
    fig.subplots_adjust(left=.25,right=.97,top=.96,bottom=.23)
    save_figure(fig,f'fig{33+lev:02d}_signed_win_margin_c{lev}')
save_csv(pd.DataFrame(win_rows),'table_paired_signed_win_margins')
print('Paired win matrices: C1/C2/C3, each paired within the same 37 participants.')


In [ ]:
# 37. Mode × complexity and participant centered order diagnostic.
# Cell centering removes additive mode-by-complexity averages, then participant means
# are removed to focus on relative changes across successive blocks.
require_columns(trials,['block_index','Tcorrect_analysis_ms','task_id'], 'trials')
if not set(pd.to_numeric(trials['block_index']).unique()) <= set(range(5)):
    raise ValueError('Unexpected block index; expected 0..4')
order_df=trials[['participant_code','trial_id','mode','level','block_index','Tcorrect_analysis_ms']].copy()
order_df['block_index']=order_df['block_index'].astype(int)
order_df['log_time']=np.log(order_df['Tcorrect_analysis_ms'].astype(float))
order_df['mode_level_mean_log']=order_df.groupby(['mode','level'],observed=True)['log_time'].transform('mean')
order_df['resid']=order_df['log_time']-order_df['mode_level_mean_log']
order_df['resid']-=order_df.groupby('participant_code')['resid'].transform('mean')
order_subj=order_df.groupby(['participant_code','block_index'])['resid'].mean().unstack('block_index').reindex(index=subjects,columns=range(5))
if order_subj.isna().any().any():raise ValueError('Each included participant needs 5 complete mode blocks')
order_est=[]
for block in range(5):
    v=order_subj[block].to_numpy(float)
    mean,lo,hi=bootstrap_stat(v,stat='mean')
    order_est.append(dict(block=block+1,n=len(v),log_relative_mean=mean,
                          log_ci_low=lo,log_ci_high=hi,
                          relative_time_pct=100*(np.exp(mean)-1),
                          ci_low_pct=100*(np.exp(lo)-1),ci_high_pct=100*(np.exp(hi)-1)))
order_results=pd.DataFrame(order_est)
save_csv(order_results,'table_block_order_adjusted')
fig,ax=newfig(6.6)
ax.axhline(0,color='#777777',lw=.7,ls='--')
x=order_results['block'].to_numpy()
y=order_results['relative_time_pct'].to_numpy()
err=np.vstack([y-order_results['ci_low_pct'].to_numpy(),order_results['ci_high_pct'].to_numpy()-y])
ax.errorbar(x,y,yerr=err,fmt='o-',color='#333333',capsize=2.5,markersize=4,lw=1.3)
ax.set_xticks(x,[str(i) for i in x]);ax.set_xlabel('Mode block position (1–5)')
ax.set_ylabel('Relative analysis time (%)')
scientific_axes(ax)
fig.text(.5,.033,'Centered by participant and mode × complexity',ha='center',fontsize=5.6)
fig.subplots_adjust(left=.20,right=.96,bottom=.26,top=.94)
save_figure(fig,'fig37_block_order_diagnostic')


In [ ]:
# 38–40. Adjusted differences among task variants a–e.
# Within each C level: log analysis time ~ fixed effects of participant, mode, block
# position and task variant. Leave-one-participant-out jackknife estimates uncertainty.
# These are associational diagnostics (no causal interpretations).

def fit_variant_effects(q,keep_mask=None):
    q=q.copy()
    if keep_mask is not None:q=q.loc[keep_mask].copy()
    q['participant_code']=q['participant_code'].astype(str)
    q['mode']=q['mode'].astype(str)
    q['block_index']=q['block_index'].astype(str)
    q['variant']=q['variant'].astype(str)
    available_subjects=sorted(q['participant_code'].unique().tolist())
    cols=[
        *[('participant_code',v) for v in available_subjects[1:]],
        *[('mode',v) for v in MODE_ORDER[1:]],
        *[('block_index',str(v)) for v in range(1,5)],
        *[('variant',v) for v in 'bcde'],
    ]
    # Constant plus reference-cell dummy coding, fixed column order even in jackknife.
    design=np.column_stack([np.ones(len(q))]+[(q[k].to_numpy()==v).astype(float) for k,v in cols])
    outcome=np.log(q['Tcorrect_analysis_ms'].to_numpy(float))
    params,resid,rank,singular=np.linalg.lstsq(design,outcome,rcond=None)
    if rank!=design.shape[1]:
        raise ValueError(f'Variant design not identifiable: rank {rank}/{design.shape[1]}; check Latin-square confounding')
    # Four reference contrasts to a + explicitly sum-center all five effects.
    vs=np.asarray([0.]+[params[1+cols.index(('variant',v))] for v in 'bcde'])
    return vs-vs.mean()

var_results=[]
for lev in LEVEL_ORDER:
    q=trials.loc[trials.level==lev,['participant_code','mode','block_index','task_id','Tcorrect_analysis_ms']].copy()
    q['variant']=q['task_id'].astype(str).str.extract(r'^C[123]([a-e])$',expand=False)
    if q['variant'].isna().any():raise ValueError('Unexpected task_id variant in C'+str(lev))
    point=fit_variant_effects(q)
    jack=[]
    for participant in subjects:
        jack.append(fit_variant_effects(q,keep_mask=q.participant_code.ne(participant)))
    jack=np.asarray(jack)
    # Standard jackknife SE of each centered log effect.
    jk_sd=np.sqrt((len(jack)-1)/len(jack)*np.sum((jack-jack.mean(axis=0))**2,axis=0))
    pct=100*(np.exp(point)-1)
    low=100*(np.exp(point-1.96*jk_sd)-1)
    high=100*(np.exp(point+1.96*jk_sd)-1)
    for i,variant in enumerate('abcde'):
        var_results.append(dict(level=lev,variant=variant,n=int((q.variant==variant).sum()),
                    log_effect_vs_mean=point[i],jackknife_se_log=jk_sd[i],
                    adjusted_relative_time_pct=pct[i],ci_low_pct=low[i],ci_high_pct=high[i]))
    fig,ax=newfig(6.4)
    yy=np.arange(5)
    ax.axvline(0,color='#777777',ls='--',lw=.7)
    ax.errorbar(pct,yy,xerr=np.vstack([pct-low,high-pct]),fmt='o',color='#333333',
                capsize=2.5,markersize=3.7,lw=.85)
    ax.set_yticks(yy,[f'Variant {v}' for v in 'abcde'])
    ax.invert_yaxis()
    ax.set_xlabel('Adjusted time vs. variant mean (%)')
    scientific_axes(ax,'x')
    fig.text(.5,.032,'Participant, mode & block adjusted; jackknife 95% CI',ha='center',fontsize=5.3)
    fig.subplots_adjust(left=.28,right=.97,bottom=.26,top=.96)
    save_figure(fig,f'fig{37+lev:02d}_variant_equivalence_c{lev}')
save_csv(pd.DataFrame(var_results),'table_variant_equivalence_adjusted')


In [ ]:
# 41–43. Empirical probability of a correct solution by the K-th attempt.
# Failures have no successful attempt and stay in the denominator.
if a['ordinal'].isna().any():raise ValueError('Attempt ordinal is missing')
a_success=a.loc[to_bool(a['correct'])].groupby('trial_id')['ordinal'].min()
trial_attempts=trials[['trial_id','participant_code','mode','level','success']].copy()
trial_attempts['first_success_ordinal']=trial_attempts['trial_id'].map(a_success)
if trial_attempts.loc[trial_attempts.success.eq(1),'first_success_ordinal'].isna().any():
    raise ValueError('Successful trial lacks a correct attempt in attempts.csv')
if trial_attempts.loc[trial_attempts.success.eq(0),'first_success_ordinal'].notna().any():
    raise ValueError('Unsuccessful trial contains a correct attempt')
max_k=int(max(1,a['ordinal'].max()))
attempt_curve=[]
for lev in LEVEL_ORDER:
    fig,ax=newfig(7.0)
    for mode in MODE_ORDER:
        q=trial_attempts.loc[(trial_attempts.level==lev)&(trial_attempts['mode']==mode)]
        first=q.first_success_ordinal.to_numpy(float)
        yy=np.asarray([100*np.mean(first<=k) for k in range(1,max_k+1)])
        for k,y in enumerate(yy,1):
            attempt_curve.append(dict(level=lev,mode=mode,attempt_ordinal=k,n=len(q),
                                      successfully_completed_pct=float(y)))
        ax.step(range(1,max_k+1),yy,where='post',color=COLORS[mode],
                ls=LINE_STYLES[mode],lw=1.35,label=mode)
    ax.set_xlim(1,max_k);ax.set_ylim(0,102)
    ax.set_xticks([k for k in [1,2,3,4,5,10,15,20,25] if k<=max_k])
    ax.set_xlabel('Maximum submitted attempts (k)')
    ax.set_ylabel('Solved by attempt k (%)')
    ax.legend(frameon=False,ncol=3,loc='lower right',columnspacing=.8)
    scientific_axes(ax)
    fig.subplots_adjust(left=.19,right=.97,bottom=.17,top=.95)
    save_figure(fig,f'fig{40+lev:02d}_success_by_attempt_c{lev}')
save_csv(pd.DataFrame(attempt_curve),'table_success_by_attempt')


In [ ]:
# 44. Speech duration vs RouterAI NVIDIA Nemotron ASR latency.
require_columns(transcriptions,['trial_id','audio_duration_ms','Tasr_ms'],'m4_transcriptions')
asr=transcriptions.loc[transcriptions.trial_id.isin(selected_ids)].copy()
asr=asr.merge(trials[['trial_id','participant_code','level','attempts','mode']],on='trial_id',
              validate='many_to_one',suffixes=('','_trial'))
asr=asr.loc[asr['mode_trial']=='M4'].copy()
asr['audio_s']=pd.to_numeric(asr.audio_duration_ms,errors='coerce')/1000
asr['asr_s']=pd.to_numeric(asr.Tasr_ms,errors='coerce')/1000
asr=asr.loc[np.isfinite(asr.audio_s)&np.isfinite(asr.asr_s)&(asr.audio_s>0)&(asr.asr_s>0)].copy()
if len(asr)<5:raise ValueError('Insufficient ASR observations')
x=asr.audio_s.to_numpy(float)
y=asr.asr_s.to_numpy(float)
# Robust Theil-Sen fit on log latency to reduce influence of API outliers.
slopes=[]
logy=np.log(y)
for i in range(len(x)):
    dx=x[i+1:]-x[i]
    valid=np.abs(dx)>1e-8
    slopes.extend(((logy[i+1:]-logy[i])[valid]/dx[valid]).tolist())
slope=float(np.median(slopes))
intercept=float(np.median(logy-slope*x))
spearman=float(pd.Series(x).rank().corr(pd.Series(y).rank()))
fig,ax=newfig(7.25)
LEVEL_COLORS={1:'#555555',2:'#0072B2',3:'#D55E00'}
for lev in LEVEL_ORDER:
    s=asr.loc[asr.level==lev]
    ax.scatter(s.audio_s,s.asr_s,s=8+5*s.attempts.clip(upper=8),
               c=LEVEL_COLORS[lev],alpha=.5,edgecolors='none',label=f'C{lev}')
xx=np.linspace(x.min(),x.max(),150)
ax.plot(xx,np.exp(intercept+slope*xx),ls='--',lw=1.5,color='black',label='Robust trend')
ax.set_yscale('log')
ax.set_xlabel('Spoken command duration (s)')
ax.set_ylabel('ASR API latency (s, log scale)')
ax.legend(frameon=False,ncol=2,loc='upper left',fontsize=6.0)
scientific_axes(ax)
fig.text(.5,.025,f'Spearman rho = {spearman:+.2f}; area reflects trial attempts',
         ha='center',fontsize=5.7)
fig.subplots_adjust(left=.21,right=.97,bottom=.24,top=.97)
save_figure(fig,'fig44_m4_audio_duration_vs_asr_latency')
save_csv(asr[['trial_id','participant_code','level','audio_s','asr_s','attempts']],
         'table_m4_asr_duration_and_latency')


In [ ]:
# 45. M5 total tool calls versus observed agent end-to-end latency.
require_columns(agent,['trial_id','Tagent_ms','tool_calls'],'agent_runs')
agent['Tagent_ms']=pd.to_numeric(agent['Tagent_ms'],errors='coerce')
agent_extended=(agent.groupby('trial_id',as_index=False).agg(
    tool_calls=('tool_calls','sum'),agent_latency_ms=('Tagent_ms','sum'))
    .merge(trials[['trial_id','participant_code','level','success']],on='trial_id',validate='one_to_one'))
if len(agent_extended)!=N_SUBJECTS*3:raise ValueError('M5 sample mismatch')
agent_extended['agent_latency_s']=agent_extended['agent_latency_ms']/1000
save_csv(agent_extended,'table_m5_tools_vs_agent_latency')
if (agent_extended['agent_latency_s']<=0).any():raise ValueError('Nonpositive agent latency')
fig,ax=newfig(7.25)
rng_jitter=np.random.default_rng(SEED+45)
for lev in LEVEL_ORDER:
    for success in (0,1):
        subset=agent_extended.loc[(agent_extended.level==lev)&(agent_extended.success==success)]
        if subset.empty:continue
        xs=subset.tool_calls.to_numpy(float)+rng_jitter.uniform(-.13,.13,len(subset))
        ax.scatter(xs,subset.agent_latency_s,s=21,alpha=.67,
                   c=LEVEL_COLORS[lev] if success else 'white',
                   edgecolors=LEVEL_COLORS[lev] if success else '#000000',
                   linewidth=.65,marker='o',label=(f'C{lev}' if success else 'Not solved'))
# Plot plain OLS association, without causal claim.
vx=agent_extended.tool_calls.to_numpy(float)
vy=agent_extended.agent_latency_s.to_numpy(float)
if np.unique(vx).size>1:
    slope,intercept=np.polyfit(vx,vy,deg=1)
    lx=np.linspace(vx.min(),vx.max(),100)
    ax.plot(lx,intercept+slope*lx,color='black',lw=1.2,ls='--',label='Linear association')
rho=float(pd.Series(vx).rank().corr(pd.Series(vy).rank()))
ax.set_xlabel('Agent tool calls per trial')
ax.set_ylabel('Cumulative agent latency (s)')
scientific_axes(ax)
handles,labels=ax.get_legend_handles_labels()
seen_names=set(); pairs=[]
for h,label in zip(handles,labels):
    if label not in seen_names:seen_names.add(label);pairs.append((h,label))
ax.legend([h for h,_ in pairs],[l for _,l in pairs],frameon=False,
          ncol=2,loc='upper left',fontsize=5.7)
fig.text(.5,.025,f'Spearman rho = {rho:+.2f}; horizontal jitter for visibility',
         ha='center',fontsize=5.5)
fig.subplots_adjust(left=.2,right=.97,bottom=.23,top=.97)
save_figure(fig,'fig45_m5_tool_calls_vs_agent_latency')
print('Six diagnostic analyses added: 12 new figures + 7 data tables.')


In [ ]:
from collections import Counter

# Check whether protocol-level factors vary among included participants.
protocol_columns=['trial_limit_seconds','attempt_limit','wording_version']
configurations=trials[protocol_columns].drop_duplicates().to_dict('records')

notes=[
    'Only complete sessions with participant codes matching ^P[0-9]{3}$ and 15 terminal M1-M5 x C1-C3 trials are analyzed.',
    'Each participant is the resampling unit; individual trials are not independent.',
    'Analysis time includes protocol-specific penalty for incomplete trials; actual correct time only for successes.',
    'CI are percentile cluster bootstraps; the interaction p-values are random sign-flip tests with Holm correction.',
    'All complexity levels follow C1 to C3 within each mode block (learning/order confound).',
    'Event samples do not measure cognitive effort, and pointer-move is sampling-rate dependent.',
    'Bubble chart is descriptive, log-scaled on time, with area reflecting average active time.',
    'Empirical cumulative success uses only observed successful completions; failures remain in the denominator.',
    'Residual observed time is not directly measured thinking time.',
    'Agent trajectories are observed tool/LLM calls only; do not infer hidden reasoning.',
    'Request waiting is measured from request_started durations and unioned to avoid double counting.',
    'ASR M4 is RouterAI NVIDIA Nemotron, not locally hosted GigaAM.',
    'Additional variant effects use participant, mode and block adjustments; jackknife CIs are exploratory.',
    'Block effects are descriptive, and cannot remove fixed C1→C2→C3 order confounding.',
    'Signed win margins compare row-versus-column speed across the same participants; ties contribute zero to the signed margin; original tie-half winning proportions remain in CSV.',
    'Speech latency vs duration and tool calls vs latency show associations, not causality.',
    'These charts are descriptive and hypothesis-guided; confirmatory claims need an analysis plan and protocol discussion.'
]
manifest={
    'source_zip':archive_path.name,
    'source_zip_sha256':archive_sha256,
    'generated_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'population':{'registry_sessions':len(registry),
                  'participant_code_pattern':'^P[0-9]{3}$',
                  'completion_filter':'registry.complete == True',
                  'completed_sessions':len(completed_registry),
                  'excluded_nonmatching_codes':int((~is_real_code).sum()),
                  'excluded_incomplete_P_sessions':int((is_real_code & ~registry.complete).sum()),
                  'excluded_sessions':int((~(is_real_code & registry.complete)).sum()),
                  'participants':N_SUBJECTS,'trials':len(trials),
                  'successful_trials':int(trials.success.sum()),
                  'incomplete_trials':int((1-trials.success).sum())},
    'protocol_configurations':configurations,
    'protocol_asr':protocol.get('manifest',{}).get('m4_asr',{}),
    'method':{'bootstrap_replicates':BOOTSTRAP_N,'signflip_permutations':PERMUTATIONS,
              'bootstrap_unit':'participant','alpha':0.05,
              'multiple_comparison_correction':'Holm across three M3/M4/M5 vs M1 C3-C1 interaction contrasts',
              'primary_endpoint':'Tcorrect_analysis_ms','physical_figure_width_cm':FIG_WIDTH_CM,
              'dpi':DPI,'seed':SEED,'figure_format':'PNG only'},
    'figures':[str(p.relative_to(OUTPUT_DIR)) for p in figure_paths],
    'tables':[str(p.relative_to(OUTPUT_DIR)) for p in table_paths],
    'limitations':notes,
}
(OUTPUT_DIR/'analysis_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf8')
# Figures and rendered tables must be PNG only (CSV is retained solely as numerical data).
assert all(p.suffix.lower()=='.png' for p in figure_paths), 'Non-PNG figure exported'
assert all(p.suffix.lower() in {'.png','.csv'} for p in table_paths), 'Unexpected table file format'
pdf_created=[p for folder in (FIG_DIR,TABLE_DIR) for p in folder.glob('*.pdf')]
if pdf_created: raise RuntimeError(f'PDF found in output folders: {pdf_created}. Remove legacy PDFs and rerun.')
print('PNG-only image export verified; PDF files: 0')
print('Completed. Sample size:',N_SUBJECTS,'participants /',len(trials),'trials.')
print('PNG figures (8 cm):',len(figure_paths), 'PNG tables:',sum(p.suffix=='.png' for p in table_paths), 'CSV:',sum(p.suffix=='.csv' for p in table_paths))
print('Output directory:',OUTPUT_DIR)
print('For the manuscript, select from figures 01–45; place other results in supplementary materials.')

## 17. Final outputs: PNG-only figures and rendered tables

All **45 figures** (`output/figures/fig01…fig45*.png`) and **four rendered tables** (`output/tables/*.png`) have a physical width of **8 cm** at **300 dpi**. No `.pdf`, `.svg`, or `.eps` image files are generated. CSV files contain **numeric tabular results**, while `output/analysis_manifest.json` records the inclusion protocol, ZIP checksum, and limits of interpretation.

Place `analytics.ipynb` beside a single `task-mining*.zip` file in `analytics/`, then choose **Run All**. Given the same archive and random `SEED`, the analyses are reproducible, except for the manifest generation timestamp.
